# Week 7: The Adaptation Decision and a Production Dataset

This notebook builds and evaluates a no-GPU adaptation for a five-class assistant intent-classification task.

**Course constraint:** no fine-tuning step is performed. The dataset and few-shot retrieval run locally. Gemini Flash-Lite is used only for the measured zero-shot and dynamic few-shot evaluation.

The committed dataset contains **200 training examples** and **50 held-out evaluation examples**. To stay within free-tier API limits, the measured model comparison uses a deterministic, label-balanced **25-example evaluation subset** (5 examples per class), as allowed by the assignment guidance. Responses are cached so rerunning the notebook does not repeat completed API calls.


In [ ]:
# Install once in Colab/Jupyter if needed:
# %pip install -q pandas sentence-transformers transformers google-genai

import os, re, json, time, pathlib, hashlib
import pandas as pd
ROOT = pathlib.Path('.')
if not (ROOT / 'week7_train.csv').exists() and (ROOT / 'week-07').exists(): ROOT = ROOT / 'week-07'
TRAIN_PATH, EVAL_PATH, CACHE_PATH = ROOT/'week7_train.csv', ROOT/'week7_eval.csv', ROOT/'week7_eval_cache.json'
train_df, eval_df = pd.read_csv(TRAIN_PATH), pd.read_csv(EVAL_PATH)
print('training rows:', len(train_df)); print('evaluation rows:', len(eval_df)); train_df.head()


## Part 1 — Dataset quality report

The examples are synthetic and created specifically for this assignment. Each row has `text` and `label`. The five labels are `set_reminder`, `get_weather`, `play_music`, `send_message`, and `get_directions`. The training and evaluation files were generated separately so held-out wording is not copied from training templates.


In [ ]:
VALID_LABELS={'set_reminder','get_weather','play_music','send_message','get_directions'}
def norm(s): return re.sub(r'\s+',' ',str(s).strip().lower())
def dupes(df): return int(df.assign(_n=df.text.map(norm)).duplicated(subset=['_n','label']).sum())
def errors(df): return int(df.text.isna().sum()+df.label.isna().sum()+df.text.fillna('').map(lambda x:not str(x).strip()).sum()+(~df.label.isin(VALID_LABELS)).sum())
leakage=set(train_df.text.map(norm)) & set(eval_df.text.map(norm))
print('train format errors:',errors(train_df)); print('eval format errors:',errors(eval_df))
print('train duplicates:',dupes(train_df)); print('eval duplicates:',dupes(eval_df)); print('train/eval exact overlap:',len(leakage))
print('train labels:',train_df.label.value_counts().sort_index().to_dict()); print('eval labels:',eval_df.label.value_counts().sort_index().to_dict())
assert len(train_df)>=200 and len(eval_df)>=50 and errors(train_df)==errors(eval_df)==dupes(train_df)==dupes(eval_df)==len(leakage)==0
assert set(train_df.label)==VALID_LABELS==set(eval_df.label); print('All dataset validation assertions passed.')


In [ ]:
from transformers import AutoTokenizer
tok=AutoTokenizer.from_pretrained('sentence-transformers/all-MiniLM-L6-v2')
def lengths(s): return [len(tok.encode(str(x),add_special_tokens=False)) for x in s]
def describe(name,vals):
    s=pd.Series(vals); print(f'{name}: min={int(s.min())}, p25={s.quantile(.25):.1f}, median={s.median():.1f}, p75={s.quantile(.75):.1f}, p95={s.quantile(.95):.1f}, max={int(s.max())}, mean={s.mean():.2f}')
describe('train token length',lengths(train_df.text)); describe('eval token length',lengths(eval_df.text))


### Data-quality interpretation

The quality gate rejects missing/blank records and unknown labels, deduplicates normalized `(text, label)` pairs, and prohibits exact text overlap between training and evaluation. Label balance is fixed at 40 training and 10 evaluation examples per class. The main limitation is that the data are synthetic; production data would need human-reviewed examples and a documented labeling guide.


## Part 2 — Dynamic few-shot adaptation

MiniLM embeds each query and all 200 training examples locally. Cosine similarity selects the four nearest training examples, which are inserted into the Gemini prompt as demonstrations. The zero-shot baseline receives the same task instruction and labels but no demonstrations.


In [ ]:
from sentence_transformers import SentenceTransformer
import numpy as np
embedder=SentenceTransformer('sentence-transformers/all-MiniLM-L6-v2')
train_embeddings=embedder.encode(train_df.text.tolist(),normalize_embeddings=True,convert_to_numpy=True,show_progress_bar=False)
def retrieve_examples(query,k=4):
    q=embedder.encode([query],normalize_embeddings=True,convert_to_numpy=True)[0]; scores=train_embeddings@q; idx=np.argsort(scores)[::-1][:k]
    rows=train_df.iloc[idx].copy(); rows['similarity']=scores[idx]; return rows[['text','label','similarity']]
q=eval_df.iloc[0].text; print('query:',q); retrieve_examples(q,4)


## Part 3 — Gemini evaluation

The measured subset is deterministic and balanced: five held-out examples per label, 25 total. Both conditions use the same Gemini Flash-Lite model and subset. Temperature is 0, and responses are cached. Set `GEMINI_API_KEY` before running.


In [ ]:
eval_25=eval_df.sort_values(['label','text']).groupby('label',group_keys=False).head(5).reset_index(drop=True)
print('measured evaluation rows:',len(eval_25)); print(eval_25.label.value_counts().sort_index().to_dict())


In [ ]:
from google import genai
from google.genai import types
MODEL='gemini-3.5-flash-lite'; LABELS=sorted(VALID_LABELS); api_key=os.environ.get('GEMINI_API_KEY'); client=genai.Client(api_key=api_key) if api_key else None
cache=json.loads(CACHE_PATH.read_text()) if CACHE_PATH.exists() else {}
def make_prompt(text,shots=None):
    p='Classify the user request into exactly one intent label. Return only the label and no explanation.\nAllowed labels: '+', '.join(LABELS)+'.\n\n'
    for s in shots or []: p+=f'User: {s["text"]}\nLabel: {s["label"]}\n\n'
    return p+f'User: {text}\nLabel:'
def normalize_prediction(text):
    if text is None:return None
    c=str(text).strip().lower().replace('`',''); m=re.search(r'(set_reminder|get_weather|play_music|send_message|get_directions)',c); return m.group(1) if m else c
def predict(prompt,condition):
    if client is None:return None
    key=hashlib.sha256((MODEL+'|'+condition+'|'+prompt).encode()).hexdigest()
    if key in cache:return cache[key]
    r=client.models.generate_content(model=MODEL,contents=prompt,config=types.GenerateContentConfig(temperature=0)); pred=normalize_prediction(r.text); cache[key]=pred
    CACHE_PATH.write_text(json.dumps(cache,indent=2,sort_keys=True)+'\n'); time.sleep(.2); return pred


In [ ]:
results=[]
if client is None: print('[API-BLOCKED] Set GEMINI_API_KEY, then rerun this cell.')
else:
    for _,r in eval_25.iterrows():
        text,gold=r.text,r.label; z=predict(make_prompt(text),'zero-shot'); shots=retrieve_examples(text,4)[['text','label']].to_dict('records'); f=predict(make_prompt(text,shots),'dynamic-few-shot')
        results.append({'text':text,'gold':gold,'zero_pred':z,'few_pred':f,'zero_correct':z==gold,'few_correct':f==gold,'retrieved_labels':[s['label'] for s in shots]})
    results_df=pd.DataFrame(results); zero_acc=results_df.zero_correct.mean(); few_acc=results_df.few_correct.mean()
    print(f'zero-shot accuracy: {zero_acc:.1%}'); print(f'dynamic few-shot accuracy: {few_acc:.1%}'); print(f'absolute change: {(few_acc-zero_acc):+.1%}'); display(results_df)


In [ ]:
if 'results_df' not in globals() or results_df.empty: print('Run the Gemini evaluation cell first.')
else:
    by_label=results_df.groupby('gold')[['zero_correct','few_correct']].mean().rename(columns={'zero_correct':'zero_shot_acc','few_correct':'few_shot_acc'}); by_label['delta']=by_label.few_shot_acc-by_label.zero_shot_acc; display(by_label)
    regressions=results_df[(results_df.zero_correct)&(~results_df.few_correct)]; worse=by_label[by_label.delta<0]
    if not worse.empty:
        lab=worse.delta.idxmin(); print('Underperforming slice:',lab); display(results_df[results_df.gold==lab]); print('Likely failure mode: retrieval-induced label bias from semantically similar demonstrations.')
    elif not regressions.empty: print('No whole label slice regressed; individual regressions:'); display(regressions)
    else: print('No regression observed in this 25-example run. Do not invent one; expand to all 50 if limits allow or rerun a pre-declared slice.')


## Part 4 — Adaptation decision memo

**Decision.** Use the simplest method that produces a measured improvement. A constrained prompt is the default if zero-shot accuracy is already high; dynamic few-shot is justified only if the executed results show a meaningful gain worth the added retrieval and prompt complexity.

**Prompting.** This task has a small fixed label set and short inputs, so a clear instruction and exact-output constraint are a strong baseline.

**Retrieval.** Retrieval alone does not produce the label; it finds similar examples. A traditional RAG knowledge pipeline would add complexity without solving a knowledge problem.

**Dynamic few-shot.** This adapts behavior without training or a GPU, but a bad neighbor can bias the answer, so the failure slice matters.

**Fine-tuning.** Fine-tuning becomes worth its cost if a much larger human-reviewed dataset shows stable recurring errors that prompting and dynamic examples cannot fix, and task volume justifies training and maintenance. It is not the current choice: the course prohibits a tuning step, the Gemini API does not currently offer general model tuning, and inference-time adaptation should be tested first.

**Do nothing.** If zero-shot meets the target and few-shot does not materially improve it, leaving the model alone is defensible.

### Fill after execution
- Zero-shot accuracy: **[RUN NOTEBOOK]**
- Dynamic few-shot accuracy: **[RUN NOTEBOOK]**
- Absolute change: **[RUN NOTEBOOK]**
- Underperforming slice: **[RUN NOTEBOOK]**
- Final choice: **[PROMPT / DYNAMIC FEW-SHOT / DO NOTHING]**


## Submission checklist

- [x] 200 training examples and 50 held-out examples
- [x] validation, deduplication, leakage, label balance, format checks, token distribution
- [x] local dynamic few-shot retrieval
- [x] Gemini Flash-Lite baseline and adapted evaluation with caching
- [x] failure-slice analysis and decision memo
- [ ] run all cells with `GEMINI_API_KEY`
- [ ] replace memo placeholders with measured results
- [ ] save executed notebook and export HTML for Canvas
